<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/voxtral_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
Mounted at /content/drive


In [2]:
!chmod a+x /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-mtmd-cli
!ls -ltha /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-mtmd-cli

-rwx------ 1 root root 105K Nov  2 11:03 /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-mtmd-cli
-rwx------ 1 root root 105K Nov  2 11:03 /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-mtmd-cli


In [5]:
import os
import re
import json
import math
import time
import shutil
import hashlib
import logging
import pathlib
import warnings

# ---------------------------------------------------------------------------
# 1. Warning Suppression & Environment Setup (Before HF / PyTorch Imports)
# ---------------------------------------------------------------------------
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module=r".*transformers.*")
warnings.filterwarnings("ignore", message=r".*rope_scaling.*")
warnings.filterwarnings("ignore", message=r".*AttentionMaskConverter.*")
warnings.filterwarnings("ignore", message=r".*masking_utils.*")
warnings.filterwarnings("ignore", message=r".*use_return_dict.*")
warnings.filterwarnings("ignore", message=r".*torch_dtype.*")
warnings.filterwarnings("ignore", message=r".*resume_download.*")

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
for _offline_var in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE"):
    os.environ.pop(_offline_var, None)

for _lg_name in (
    "huggingface_hub",
    "huggingface_hub.utils._http",
    "transformers",
    "transformers.configuration_utils",
    "transformers.modeling_attn_mask_utils",
):
    logging.getLogger(_lg_name).setLevel(logging.ERROR)

## audio-file

In [3]:
import array
import os
import subprocess
import urllib.request
import wave

WAV_URL = (
    "https://upload.wikimedia.org/wikipedia/commons/3/38/"
    "President_Barack_Obama_Recites_President_Abraham_Lincoln%27s_Gettysburg_Address.wav"
)
RAW_WAV = "/content/gettysburg_raw.wav"
OUT_WAV = "/content/better_speech_16k.wav"

req = urllib.request.Request(
    WAV_URL,
    headers={"User-Agent": "SOMALA-Voxtral-Eval/1.0 (frank.morales@sovereign-machine-lab.ai)"},
)
with urllib.request.urlopen(req, timeout=60) as resp:
    data = resp.read()

with open(RAW_WAV, "wb") as f:
    f.write(data)

subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-i",
        RAW_WAV,
        "-ac",
        "1",
        "-ar",
        "16000",
        "-c:a",
        "pcm_s16le",
        OUT_WAV,
    ],
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

with wave.open(OUT_WAV, "rb") as wf:
    sr = wf.getframerate()
    n_frames = wf.getnframes()
    samples = array.array("h", wf.readframes(n_frames))
    peak = max((abs(x) for x in samples), default=0)

print(f"Saved: {OUT_WAV} | sr={sr} Hz | duration={n_frames / float(sr):.2f}s | peak_int16={peak}")

Saved: /content/better_speech_16k.wav | sr=16000 Hz | duration=105.50s | peak_int16=32768


## inference-case1

In [8]:
import os
import re
import subprocess
from google.colab import userdata
from huggingface_hub import hf_hub_download, login

HF_TOKEN = userdata.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)

REPO_ID = "frankmorales2020/voxtral-mini-4b-realtime-2602-topo-triton-GGUF"
MODEL_FILENAME = "voxtral-mini-4b-realtime-2602-topo-triton-bf16.gguf"
MMPROJ_FILENAME = "mmproj-voxtral-mini-4b-realtime-2602-f16.gguf"

MTMD_CLI = "/content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-mtmd-cli"
WAV_PATH = "/content/better_speech_16k.wav"
CACHE_DIR = "/content/hf_gguf_test"

model_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=MODEL_FILENAME,
    local_dir=CACHE_DIR,
    token=HF_TOKEN,
)
mmproj_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=MMPROJ_FILENAME,
    local_dir=CACHE_DIR,
    token=HF_TOKEN,
)

env = os.environ.copy()
build_lib_dirs = [
    "/content/drive/MyDrive/topogpt/llama.cpp/build/bin",
    "/content/drive/MyDrive/topogpt/llama.cpp/build/src",
    "/content/drive/MyDrive/topogpt/llama.cpp/build/common",
    "/content/drive/MyDrive/topogpt/llama.cpp/build/ggml/src",
]
existing_ld = env.get("LD_LIBRARY_PATH", "")
env["LD_LIBRARY_PATH"] = ":".join(build_lib_dirs + ([existing_ld] if existing_ld else []))

cmd = [
    MTMD_CLI,
    "-m",
    model_path,
    "--mmproj",
    mmproj_path,
    "--audio",
    WAV_PATH,
    "-p",
    "Transcribe the speech in this audio accurately.",
]

print("Downloaded model:", model_path)
print("Downloaded mmproj:", mmproj_path)
print("Running:", " ".join(cmd))

proc = subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=900)
print("Exit code:", proc.returncode)
print("--- STDOUT ---")
print(proc.stdout)

noisy_patterns = (
    r"^\d+\.\d+\.\d+\.\d+\s+I\s+",                                    # info lines (threadpool, mtmd_cli_context, loading model)
    r"^\d+\.\d+\.\d+\.\d+\s+W\s+init_audio:",                          # experimental audio warning
    r"^\s*https://github\.com/ggml-org/llama\.cpp/discussions/13759",
    r"^\[SYSTEM_PROMPT\]",                                             # chat template example line
)
noisy_re = re.compile("|".join(noisy_patterns))

if proc.stderr:
    if proc.returncode != 0:
        print("--- STDERR ---")
        print(proc.stderr[-2000:])
    else:
        filtered_err = [
            line for line in proc.stderr.splitlines() if not noisy_re.search(line)
        ]
        if filtered_err:
            print("--- STDERR (filtered) ---")
            print("\n".join(filtered_err))

# Parse BF16 timing metrics and compare with Q8_0 baseline on /content/better_speech_16k.wav
q8_metrics = {
    "size_gb": 3.66,
    "total_s": 203.52,
    "rt_factor": 0.52,
    "encoder_s": 71.73,
    "decoder_s": 131.77,
    "ms_per_80ms": 153.0,
}

combined_logs = proc.stdout + ("\n" + proc.stderr if proc.stderr else "")
bf16_size_gb = os.path.getsize(model_path) / (1024**3)
m_total = re.search(
    r"run_audio_stream:\s+(\d+)\s+tokens,\s+([\d.]+)\s+s audio in\s+([\d.]+)\s+s\s+\(([\d.]+)x realtime\)",
    combined_logs,
)
m_parts = re.search(
    r"run_audio_stream:\s+encoder\s+([\d.]+)\s+s,\s+decoder\s+([\d.]+)\s+s,\s+([\d.]+)\s+ms per 80 ms step",
    combined_logs,
)

if m_total and m_parts:
    bf16_total_s = float(m_total.group(3))
    bf16_rt = float(m_total.group(4))
    bf16_enc_s = float(m_parts.group(1))
    bf16_dec_s = float(m_parts.group(2))
    bf16_step_ms = float(m_parts.group(3))

    print("\n=== Runtime Comparison (105.50 s audio) ===")
    print(f"File size:     Q8_0 = {q8_metrics['size_gb']:.2f} GB   |  BF16 = {bf16_size_gb:.2f} GB")
    print(f"Encoder (F16): Q8_0 = {q8_metrics['encoder_s']:.2f} s    |  BF16 = {bf16_enc_s:.2f} s")
    print(f"Decoder:       Q8_0 = {q8_metrics['decoder_s']:.2f} s   |  BF16 = {bf16_dec_s:.2f} s")
    print(f"Total time:    Q8_0 = {q8_metrics['total_s']:.2f} s   |  BF16 = {bf16_total_s:.2f} s")
    print(f"Speed ratio:   Q8_0 = {q8_metrics['rt_factor']:.2f}x RT    |  BF16 = {bf16_rt:.2f}x RT")
    print(f"Step latency:  Q8_0 = {q8_metrics['ms_per_80ms']:.1f} ms/80ms |  BF16 = {bf16_step_ms:.1f} ms/80ms")

Downloaded model: /content/hf_gguf_test/voxtral-mini-4b-realtime-2602-topo-triton-bf16.gguf
Downloaded mmproj: /content/hf_gguf_test/mmproj-voxtral-mini-4b-realtime-2602-f16.gguf
Running: /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-mtmd-cli -m /content/hf_gguf_test/voxtral-mini-4b-realtime-2602-topo-triton-bf16.gguf --mmproj /content/hf_gguf_test/mmproj-voxtral-mini-4b-realtime-2602-f16.gguf --audio /content/better_speech_16k.wav -p Transcribe the speech in this audio accurately.
Exit code: 0
--- STDOUT ---
Transcribing 105.5 s of audio, fed in chunks of 320 ms

Four score and seven years ago our fathers brought forth upon this continent a new nation conceived in liberty and dedicated to the proposition that all men are created equal now we are engaged in a great civil war testing whether that nation or any nation so conceived and so dedicated can long endure we are met on a great battlefield of that war we have come to dedicate a portion of it as a final resting place for

## inference-case2

In [9]:
import hashlib
import json
import os
import re
import subprocess
from google.colab import userdata
from huggingface_hub import hf_hub_download, login

# --- Configurable Parameters ---
USER_OR_NAME = "frankmorales2020"
REPO_ID = f"{USER_OR_NAME}/voxtral-mini-4b-realtime-2602-topo-triton-GGUF"
MODEL_FILENAMES = [
    "voxtral-mini-4b-realtime-2602-topo-triton-Q8_0.gguf",
    "voxtral-mini-4b-realtime-2602-topo-triton-bf16.gguf",
]
MMPROJ_FILENAME = "mmproj-voxtral-mini-4b-realtime-2602-f16.gguf"

MTMD_CLI = "/content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-mtmd-cli"
WAV_PATH = "/content/better_speech_16k.wav"
CACHE_DIR = "/content/hf_gguf_test"
PROMPT = "Transcribe the speech in this audio accurately."

N_GPU_LAYERS = -1          # 0 for CPU-only, -1 for full offload, or an int for partial
SEED = 123
TEMPERATURE = 0.0
REPEAT_PENALTY = 1.1
TIMEOUT_S = 900
SAVE_JSON_PATH = "/content/drive/MyDrive/topogpt/voxtral_q8_vs_bf16.json"
# -------------------------------

HF_TOKEN = userdata.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)

env = os.environ.copy()
build_lib_dirs = [
    "/content/drive/MyDrive/topogpt/llama.cpp/build/bin",
    "/content/drive/MyDrive/topogpt/llama.cpp/build/src",
    "/content/drive/MyDrive/topogpt/llama.cpp/build/common",
    "/content/drive/MyDrive/topogpt/llama.cpp/build/ggml/src",
]
existing_ld = env.get("LD_LIBRARY_PATH", "")
env["LD_LIBRARY_PATH"] = ":".join(build_lib_dirs + ([existing_ld] if existing_ld else []))

mmproj_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=MMPROJ_FILENAME,
    local_dir=CACHE_DIR,
    token=HF_TOKEN,
)

noisy_re = re.compile(
    "|".join((
        r"^warning:\s+(no usable GPU found|one possible reason|consult docs/build\.md)",
        r"^\d+\.\d+\.\d+\.\d+\s+I\s+",
        r"^\d+\.\d+\.\d+\.\d+\s+W\s+init_audio:",
        r"^\s*https://github\.com/ggml-org/llama\.cpp/discussions/13759",
        r"^\[SYSTEM_PROMPT\]",
    ))
)

def run_checkpoint(model_filename: str) -> dict:
    model_path = hf_hub_download(
        repo_id=REPO_ID,
        filename=model_filename,
        local_dir=CACHE_DIR,
        token=HF_TOKEN,
    )

    cmd = [
        MTMD_CLI,
        "-m", model_path,
        "--mmproj", mmproj_path,
        "--audio", WAV_PATH,
        "-p", PROMPT,
    ]
    if N_GPU_LAYERS is not None:
        cmd.extend(["-ngl", str(N_GPU_LAYERS)])
    if SEED is not None:
        cmd.extend(["--seed", str(SEED)])
    if TEMPERATURE is not None:
        cmd.extend(["--temp", str(TEMPERATURE)])
    if REPEAT_PENALTY is not None:
        cmd.extend(["--repeat-penalty", str(REPEAT_PENALTY)])

    print(f"\n=== Running {model_filename} ===")
    proc = subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=TIMEOUT_S)

    stdout_sha256 = hashlib.sha256(proc.stdout.encode("utf-8")).hexdigest()
    size_gb = round(os.path.getsize(model_path) / (1024**3), 4)

    if proc.stderr:
        if proc.returncode != 0:
            print("--- STDERR ---")
            print(proc.stderr[-2000:])
        else:
            filtered_err = [ln for ln in proc.stderr.splitlines() if not noisy_re.search(ln)]
            if filtered_err:
                print("--- STDERR (filtered) ---")
                print("\n".join(filtered_err))

    combined_logs = proc.stdout + ("\n" + proc.stderr if proc.stderr else "")
    m_total = re.search(
        r"run_audio_stream:\s+(\d+)\s+tokens,\s+([\d.]+)\s+s audio in\s+([\d.]+)\s+s\s+\(([\d.]+)x realtime\)",
        combined_logs,
    )
    m_parts = re.search(
        r"run_audio_stream:\s+encoder\s+([\d.]+)\s+s,\s+decoder\s+([\d.]+)\s+s,\s+([\d.]+)\s+ms per 80 ms step",
        combined_logs,
    )

    metrics = {
        "size_gb": size_gb,
        "stdout_sha256": stdout_sha256,
        "returncode": proc.returncode,
    }
    if m_total and m_parts:
        metrics.update({
            "tokens": int(m_total.group(1)),
            "audio_s": float(m_total.group(2)),
            "total_s": float(m_total.group(3)),
            "rt_factor": float(m_total.group(4)),
            "encoder_s": float(m_parts.group(1)),
            "decoder_s": float(m_parts.group(2)),
            "ms_per_80ms": float(m_parts.group(3)),
        })

    print(f"Exit code: {proc.returncode} | SHA-256: {stdout_sha256}")
    if "total_s" in metrics:
        print(
            f"Tokens: {metrics['tokens']} | Total: {metrics['total_s']:.2f}s "
            f"({metrics['rt_factor']:.2f}x RT) | Enc: {metrics['encoder_s']:.2f}s | "
            f"Dec: {metrics['decoder_s']:.2f}s | Step: {metrics['ms_per_80ms']:.1f} ms/80ms"
        )
    return metrics

results = {fn: run_checkpoint(fn) for fn in MODEL_FILENAMES}

q8 = results.get("voxtral-mini-4b-realtime-2602-topo-triton-Q8_0.gguf", {})
bf16 = results.get("voxtral-mini-4b-realtime-2602-topo-triton-bf16.gguf", {})

if "total_s" in q8 and "total_s" in bf16:
    print(f"\n=== Side-by-Side Comparison ({bf16['audio_s']:.2f} s audio) ===")
    print(f"File size:     Q8_0 = {q8['size_gb']:.2f} GB   |  BF16 = {bf16['size_gb']:.2f} GB")
    print(f"Encoder (F16): Q8_0 = {q8['encoder_s']:.2f} s    |  BF16 = {bf16['encoder_s']:.2f} s")
    print(f"Decoder:       Q8_0 = {q8['decoder_s']:.2f} s   |  BF16 = {bf16['decoder_s']:.2f} s")
    print(f"Total time:    Q8_0 = {q8['total_s']:.2f} s   |  BF16 = {bf16['total_s']:.2f} s")
    print(f"Speed ratio:   Q8_0 = {q8['rt_factor']:.2f}x RT    |  BF16 = {bf16['rt_factor']:.2f}x RT")
    print(f"Step latency:  Q8_0 = {q8['ms_per_80ms']:.1f} ms/80ms |  BF16 = {bf16['ms_per_80ms']:.1f} ms/80ms")
    print(f"SHA-256 match: {q8['stdout_sha256'] == bf16['stdout_sha256']}")

if SAVE_JSON_PATH:
    os.makedirs(os.path.dirname(SAVE_JSON_PATH), exist_ok=True)
    with open(SAVE_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(
            {
                "repo_id": REPO_ID,
                "mmproj_file": MMPROJ_FILENAME,
                "wav_path": WAV_PATH,
                "n_gpu_layers": N_GPU_LAYERS,
                "seed": SEED,
                "temperature": TEMPERATURE,
                "repeat_penalty": REPEAT_PENALTY,
                "results": results,
            },
            f,
            indent=2,
        )
    print(f"\nSaved JSON report to: {SAVE_JSON_PATH}")

voxtral-mini-4b-realtime-2602-topo-trito(…): reconstructing file:   0%|          |  0.00B / 3.66GB            

voxtral-mini-4b-realtime-2602-topo-trito(…): downloading bytes:           |  0.00B            


=== Running voxtral-mini-4b-realtime-2602-topo-triton-Q8_0.gguf ===
Exit code: 0 | SHA-256: c3b239821f006e717d44000b49faf398ffd149e4bf9b8455a913ec05a5f2f7f1
Tokens: 1330 | Total: 211.91s (0.50x RT) | Enc: 73.65s | Dec: 138.25s | Step: 159.3 ms/80ms

=== Running voxtral-mini-4b-realtime-2602-topo-triton-bf16.gguf ===
Exit code: 0 | SHA-256: 42eb1b6bfbffe55059e421adeea0d0cba3c7785f10df0b05d84119045a5ac108
Tokens: 1330 | Total: 276.60s (0.38x RT) | Enc: 71.65s | Dec: 204.94s | Step: 208.0 ms/80ms

=== Side-by-Side Comparison (105.50 s audio) ===
File size:     Q8_0 = 3.41 GB   |  BF16 = 6.40 GB
Encoder (F16): Q8_0 = 73.65 s    |  BF16 = 71.65 s
Decoder:       Q8_0 = 138.25 s   |  BF16 = 204.94 s
Total time:    Q8_0 = 211.91 s   |  BF16 = 276.60 s
Speed ratio:   Q8_0 = 0.50x RT    |  BF16 = 0.38x RT
Step latency:  Q8_0 = 159.3 ms/80ms |  BF16 = 208.0 ms/80ms
SHA-256 match: False

Saved JSON report to: /content/drive/MyDrive/topogpt/voxtral_q8_vs_bf16.json
